# ✨ Dream Canvas - Colab Setup

Welcome! This notebook allows you to run **Dream Canvas** on Google Colab's free GPUs.

### Prerequisite:
1. **Ngrok Token**: You need a free [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken) account for the public URL.
2. **Files**: The code below will try to clone the project from GitHub automatically. If that fails, you can upload files manually.

## 1. Check GPU Status
Creating art requires power! Let's make sure you have a GPU enabled.

In [ ]:
import torch
try:
    if torch.cuda.is_available():
        print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
        print("   You are ready to go!")
    else:
        print("⚠️ NO GPU DETECTED!")
        print("   Please go to 'Runtime > Change runtime type' in the top menu and select 'T4 GPU'.")
except ImportError:
    print("⚠️ Torch not installed yet. Run the next cell first then check back here if you want.")

## 2. Setup Project Files (Auto-Clone)
If you haven't uploaded files manually, we will try to clone them from GitHub.

In [ ]:
import os
# ---------------------------
# CONFIGURATION
# ---------------------------
REPO_URL = "https://github.com/arundevkm-official/image-gen.git" # <--- REPLACE WITH YOUR REPO URL

# Check if files are missing
if not os.path.exists("main.py") or not os.path.exists("static"):
    print(f"📂 Files not found locally. Cloning from {REPO_URL}...")
    !git clone {REPO_URL} temp_repo
    !mv temp_repo/* .
    !rm -rf temp_repo
    print("✅ Project files cloned successfully!")
else:
    print("✅ Project files found locally (uploaded manually).")

## 3. Install Dependencies

In [ ]:
!pip install fastapi uvicorn pyngrok diffusers transformers accelerate torch requests

## 4. Setup Ngrok & Run Server
Enter your Ngrok Authtoken below. The app will generate a public URL for you.

In [ ]:
import os
import threading
import subprocess
import time
import requests
from pyngrok import ngrok, conf

# ---------------------------
# CONFIGURATION
# ---------------------------
NGROK_TOKEN = "37HYe6vsS9NpYZoOHcB7lp3y0Af_WJLdrRtpJ75ARB5WCJFK"  # <--- REPLACE THIS
NGROK_DOMAIN = ""                      # <--- OPTIONAL: Enter your static domain

PORT = 8000

# ---------------------------
# PRE-FLIGHT CHECKS
# ---------------------------
if not os.path.exists("static"):
    print("❌ ERROR: 'static' folder not found!")
    raise FileNotFoundError("Missing static folder")

if not os.path.exists("main.py"):
    print("❌ ERROR: 'main.py' not found!")
    raise FileNotFoundError("Missing main.py")

# ---------------------------
# AGGRESSIVE CLEANUP
# ---------------------------
print("Cleaning up previous sessions...")
os.system("pkill ngrok")
os.system("pkill uvicorn")
time.sleep(2)

# Authenticate
conf.get_default().auth_token = NGROK_TOKEN

# ---------------------------
# START SERVER WITH LOG STREAMING
# ---------------------------

def tail_logs(filename, duration=10):
    if os.path.exists(filename):
        os.system(f"tail -n 3 {filename}")

run_log = open("server.log", "w")
subprocess.Popen(
    ["uvicorn", "main:app", "--port", str(PORT), "--host", "127.0.0.1"],
    stdout=run_log,
    stderr=run_log,
    bufsize=0
)

print("Starting server...")
print("Waiting for server to load model (this may take up to 5 minutes)...")
print("Streaming server logs below so you can see progress:\n")

server_ready = False
max_retries = 150 # 5 minutes

for i in range(max_retries):
    try:
        requests.get(f"http://127.0.0.1:{PORT}")
        server_ready = True
        break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
        if i % 5 == 0:
            tail_logs("server.log")

if not server_ready:
    print("\n❌ Server failed to start! Full logs:")
    os.system("cat server.log")
else:
    print("\n✅ Server is ready!")
    try:
        if NGROK_DOMAIN:
            public_url = ngrok.connect(f"127.0.0.1:{PORT}", domain=NGROK_DOMAIN).public_url
        else:
            public_url = ngrok.connect(f"127.0.0.1:{PORT}").public_url
        print(f"\n🚀 Dream Canvas is LIVE at: {public_url}\n")
    except Exception as e:
        print(f"Ngrok error: {e}")